# U04 · PyTorch 练习

4 道题，循序渐进。最后一题用 `nn.Module` 重写 U3 的线性回归。


## 练习 4.1 · Tensor 基础

下列每题写一行 PyTorch 代码完成。


In [3]:
import torch

# 1) 创建一个 shape=(3,4) 全 1 的 float tensor
a = torch.ones(3,4, dtype=torch.float)

# 2) 创建一个 shape=(2,3,4) 的标准正态随机 tensor，固定随机种子 0
torch.manual_seed(0)
b = torch.randn(2,3,4)

# 3) 把 a 形状改成 (4, 3)
c = a.reshape(4,3)

# 4) 把 b 沿 axis=1 求平均（保留维度），结果 shape 应该是 (2, 1, 4)
d = b.mean(dim=1, keepdim=True)

# 5) 把 numpy 数组 [1,2,3] 转成 tensor
import numpy as np
arr = np.array([1, 2, 3])
e = torch.from_numpy(arr).float()

# 自检
print(a.shape if a is not None else 'TODO 1')
print(b.shape if b is not None else 'TODO 2')
print(c.shape if c is not None else 'TODO 3')
print(d.shape if d is not None else 'TODO 4')
print(e if e is not None else 'TODO 5')


torch.Size([3, 4])
torch.Size([2, 3, 4])
torch.Size([4, 3])
torch.Size([2, 1, 4])
tensor([1., 2., 3.])


## 练习 4.2 · autograd 手算 vs 自动算

函数 $f(x, y) = x^2 y + 3y$，在 $(x=2, y=4)$ 处。

1. 先**手算** $\partial f/\partial x$ 和 $\partial f/\partial y$
2. 用 PyTorch autograd 验证


In [7]:
# 手算答案先写在这里：
# df/dx = 2 * y * x
# df/dy = x**2 + 3

import torch

# TODO: 用 autograd 验证
x = torch.tensor(2.0, requires_grad=True)  # tensor(2.0, requires_grad=True)
y = torch.tensor(4.0, requires_grad=True)  # tensor(4.0, requires_grad=True)
f = x**2 * y + 3*y  # x**2 * y + 3*y
f.backward()

print('df/dx =', x.grad)
print('df/dy =', y.grad)


df/dx = tensor(16.)
df/dy = tensor(7.)


## 练习 4.3 · 用 autograd 重写 3.4（不用 nn.Module）

目标：把 U3 练习 3.4 的线性回归用 PyTorch 重写，**手动管理 w, b**，但用 autograd 算梯度。

对比体会：你**不用再手推 dw 和 db** 了。


In [11]:
import torch
import torch.nn.functional as F
torch.manual_seed(42)


# 数据（和 3.4 一样的关系：y = 2x + 3 + 噪声）
N = 100
x = torch.empty(N).uniform_(-5, 5)
y = 2 * x + 3 + torch.randn(N) * 0.5

# 初始化参数（注意 requires_grad=True）
w = torch.tensor(0.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)

lr = 0.01

for epoch in range(500):
    
    # TODO 1: 前向 y_pred = ?
    y_pred = x * w + b

    # TODO 2: loss = MSE
    loss = F.mse_loss(y_pred, y)

    # TODO 3: 反向传播
    loss.backward()

    # TODO 4: 用 torch.no_grad() 更新参数（不要追踪更新本身）
    with torch.no_grad():
        w -= lr * w.grad
        b -= lr * b.grad
        w.grad.zero_()
        b.grad.zero_()        


    if epoch % 50 == 0 and loss is not None:
        print(f'epoch {epoch:3d}  loss={loss.item():.4f}  w={w.item():.3f}  b={b.item():.3f}')

print(f'\n最终 w = {w.item():.3f} (理论 2.0)')
print(f'最终 b = {b.item():.3f} (理论 3.0)')


epoch   0  loss=44.7097  w=0.350  b=0.066
epoch  50  loss=1.3303  w=1.999  b=1.941
epoch 100  loss=0.3213  w=1.985  b=2.613
epoch 150  loss=0.1866  w=1.980  b=2.859
epoch 200  loss=0.1686  w=1.978  b=2.948
epoch 250  loss=0.1662  w=1.977  b=2.981
epoch 300  loss=0.1658  w=1.977  b=2.993
epoch 350  loss=0.1658  w=1.977  b=2.997
epoch 400  loss=0.1658  w=1.977  b=2.999
epoch 450  loss=0.1658  w=1.977  b=3.000

最终 w = 1.977 (理论 2.0)
最终 b = 3.000 (理论 3.0)


## 练习 4.4 🌟 · 用 nn.Module + optimizer 重写（本单元过关标准）

把 4.3 升级成「工业写法」：用 `nn.Linear` + `nn.MSELoss` + `torch.optim.SGD`。

训练完应有 `w ≈ 2.0, b ≈ 3.0`，并画出 loss 曲线。


In [12]:
import torch
import torch.nn as nn
torch.manual_seed(0)

# 数据：注意 nn.Linear 期望输入是 2D (batch, features)
N = 100
x = torch.empty(N, 1).uniform_(-5, 5)        # (100, 1)
y = 2 * x + 3 + torch.randn(N, 1) * 0.5      # (100, 1)

# TODO 1: 创建模型 nn.Linear(1, 1)
model = nn.Linear(1, 1)

# TODO 2: 创建 loss 函数 nn.MSELoss()
loss_fn = nn.MSELoss()

# TODO 3: 创建优化器 torch.optim.SGD(model.parameters(), lr=0.05)
optimizer = torch.optim.SGD(model.parameters(), lr=0.05)

loss_history = []

for epoch in range(200):
    # TODO 4: 五步标准训练循环
    optimizer.zero_grad()
    y_pred = model(x)
    loss = loss_fn(y_pred, y)
    loss.backward()
    optimizer.step()
    loss_history.append(loss.item())
    pass

# 自检
if model is not None:
    print('w =', model.weight.item(), '(理论 2.0)')
    print('b =', model.bias.item(),   '(理论 3.0)')


w = 1.9657071828842163 (理论 2.0)
b = 2.9946281909942627 (理论 3.0)


In [ ]:
# 可视化 loss 曲线
import matplotlib.pyplot as plt
if loss_history:
    plt.plot(loss_history)
    plt.xlabel('epoch'); plt.ylabel('loss')
    plt.title('Loss curve')
    plt.show()


---
## ✅ 过关标准

- [ ] 4.1 五个 tensor 创建/操作都对
- [ ] 4.2 手算和 autograd 结果一致
- [ ] 4.3 用 autograd 训练出 w≈2, b≈3
- [ ] 4.4 nn.Module 版训练出 w≈2, b≈3，loss 曲线下降 🎯

全部跑通告诉我 **「U4 做完了」**，进入 **U05 · 神经网络与反向传播实战**。

🔥 U4 最重要的认知：**训练循环模板 5 步，从此固定不变。**
